# ChurnSight - Data Preprocessing
**Project:** Customer Churn Prediction with Explainable AI  
**Dataset:** IBM Telco Customer Churn  
**Author:** Rishitha

## 1. Setup and Load Cleaned Data
Loading libraries and re-applying the cleaning steps from EDA.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('telco.csv')

# Re-apply cleaning from EDA
drop_cols = [
    'Customer ID', 'Total Revenue',
    'Country', 'State', 'Quarter',
    'Churn Category', 'Churn Reason',
    'Customer Status', 'Churn Score', 'CLTV',
    'City', 'Zip Code', 'Latitude', 'Longitude', 'Population'
]
df = df.drop(columns=drop_cols)

# Fill missing values
df['Internet Type'] = df['Internet Type'].fillna('No Internet')
df['Offer'] = df['Offer'].fillna('No Offer')

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nChurn distribution:")
print(df['Churn Label'].value_counts())

Shape: (7043, 35)
Missing values: 0

Churn distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


## 2. Encoding Target Variable
Converting the Churn Label from Yes/No text to binary 1/0 for modelling.

In [2]:
# Encode target variable
df['Churn Binary'] = (df['Churn Label'] == 'Yes').astype(int)

# Drop original text target
df = df.drop(columns=['Churn Label'])

print("Target encoding done:")
print(df['Churn Binary'].value_counts())
print(f"\n0 = Not Churned, 1 = Churned")
print(f"Shape: {df.shape}")

Target encoding done:
Churn Binary
0    5174
1    1869
Name: count, dtype: int64

0 = Not Churned, 1 = Churned
Shape: (7043, 35)


## 3. Identifying Categorical Columns
Separating categorical columns by their type to decide the correct encoding strategy.

In [3]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Total categorical columns: {len(cat_cols)}\n")

for col in cat_cols:
    unique_vals = df[col].unique().tolist()
    print(f"{col}")
    print(f"  Unique count: {df[col].nunique()}")
    print(f"  Values: {unique_vals}")
    print()

Total categorical columns: 22

Gender
  Unique count: 2
  Values: ['Male', 'Female']

Under 30
  Unique count: 2
  Values: ['No', 'Yes']

Senior Citizen
  Unique count: 2
  Values: ['Yes', 'No']

Married
  Unique count: 2
  Values: ['No', 'Yes']

Dependents
  Unique count: 2
  Values: ['No', 'Yes']

Referred a Friend
  Unique count: 2
  Values: ['No', 'Yes']

Offer
  Unique count: 6
  Values: ['No Offer', 'Offer E', 'Offer D', 'Offer C', 'Offer B', 'Offer A']

Phone Service
  Unique count: 2
  Values: ['No', 'Yes']

Multiple Lines
  Unique count: 2
  Values: ['No', 'Yes']

Internet Service
  Unique count: 2
  Values: ['Yes', 'No']

Internet Type
  Unique count: 4
  Values: ['DSL', 'Fiber Optic', 'Cable', 'No Internet']

Online Security
  Unique count: 2
  Values: ['No', 'Yes']

Online Backup
  Unique count: 2
  Values: ['No', 'Yes']

Device Protection Plan
  Unique count: 2
  Values: ['Yes', 'No']

Premium Tech Support
  Unique count: 2
  Values: ['No', 'Yes']

Streaming TV
  Unique co

## 4. Encoding Categorical Variables
- **Binary columns** (Yes/No): mapped directly to 1/0
- **Ordinal columns** (natural order exists): label encoded preserving order
- **Nominal columns** (no order): one-hot encoded

In [4]:
# Binary encoding - Yes/No columns
binary_cols = [
    'Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents',
    'Referred a Friend', 'Phone Service', 'Multiple Lines',
    'Internet Service', 'Online Security', 'Online Backup',
    'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
    'Streaming Movies', 'Streaming Music', 'Unlimited Data',
    'Paperless Billing'
]

for col in binary_cols:
    if col == 'Gender':
        df[col] = df[col].map({'Male': 1, 'Female': 0})
    else:
        df[col] = df[col].map({'Yes': 1, 'No': 0})

# Ordinal encoding - Contract (natural order exists)
contract_order = {'Month-to-Month': 0, 'One Year': 1, 'Two Year': 2}
df['Contract'] = df['Contract'].map(contract_order)

# One-hot encoding - nominal columns with no order
nominal_cols = ['Offer', 'Internet Type', 'Payment Method']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

print(f"Shape after encoding: {df.shape}")
print(f"\nAll columns after encoding:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes.value_counts())

Shape after encoding: (7043, 45)

All columns after encoding:
['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score', 'Churn Binary', 'Offer_No Offer', 'Offer_Offer A', 'Offer_Offer B', 'Offer_Offer C', 'Offer_Offer D', 'Offer_Offer E', 'Internet Type_Cable', 'Internet Type_DSL', 'Internet Type_Fiber Optic', 'Internet Type_No Internet', 'Payment Method_Bank Withdrawal', 'Payment Method_Credit Card', 'Payment Method_Mailed Check']

Data types:
int6

In [5]:
# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

print(f"Bool columns converted: {bool_cols}")
print(f"\nData types after fix:")
print(df.dtypes.value_counts())
print(f"\nSample of encoded data (first 3 rows):")
print(df.head(3).to_string())

Bool columns converted: ['Offer_No Offer', 'Offer_Offer A', 'Offer_Offer B', 'Offer_Offer C', 'Offer_Offer D', 'Offer_Offer E', 'Internet Type_Cable', 'Internet Type_DSL', 'Internet Type_Fiber Optic', 'Internet Type_No Internet', 'Payment Method_Bank Withdrawal', 'Payment Method_Credit Card', 'Payment Method_Mailed Check']

Data types after fix:
int64      40
float64     5
Name: count, dtype: int64

Sample of encoded data (first 3 rows):
   Gender  Age  Under 30  Senior Citizen  Married  Dependents  Number of Dependents  Referred a Friend  Number of Referrals  Tenure in Months  Phone Service  Avg Monthly Long Distance Charges  Multiple Lines  Internet Service  Avg Monthly GB Download  Online Security  Online Backup  Device Protection Plan  Premium Tech Support  Streaming TV  Streaming Movies  Streaming Music  Unlimited Data  Contract  Paperless Billing  Monthly Charge  Total Charges  Total Refunds  Total Extra Data Charges  Total Long Distance Charges  Satisfaction Score  Churn Binary 

## 5. Feature and Target Split
Separating the dataset into features (X) and target (y) before scaling and splitting.

In [6]:
# Separate features and target
X = df.drop(columns=['Churn Binary'])
y = df['Churn Binary']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nFeature columns ({len(X.columns)}):")
print(X.columns.tolist())

Features shape (X): (7043, 44)
Target shape (y): (7043,)

Target distribution:
Churn Binary
0    5174
1    1869
Name: count, dtype: int64

Feature columns (44):
['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score', 'Offer_No Offer', 'Offer_Offer A', 'Offer_Offer B', 'Offer_Offer C', 'Offer_Offer D', 'Offer_Offer E', 'Internet Type_Cable', 'Internet Type_DSL', 'Internet Type_Fiber Optic', 'Internet Type_No Internet', 'Payment Method_Bank Withdraw

## 6. Train Test Split
Splitting data into 80% training and 20% testing before scaling to prevent data leakage.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # ensures same churn ratio in both splits
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"Churn rate in train: {y_train.mean()*100:.1f}%")

print(f"\nTesting target distribution:")
print(y_test.value_counts())
print(f"Churn rate in test: {y_test.mean()*100:.1f}%")

Training set: (5634, 44)
Testing set: (1409, 44)

Training target distribution:
Churn Binary
0    4139
1    1495
Name: count, dtype: int64
Churn rate in train: 26.5%

Testing target distribution:
Churn Binary
0    1035
1     374
Name: count, dtype: int64
Churn rate in test: 26.5%


## 7. Feature Scaling
Scaling numerical features using StandardScaler. Fitted on training data only to prevent data leakage into the test set.

In [8]:
# Only scale numerical columns, not binary or one-hot encoded columns
scale_cols = [
    'Age', 'Number of Dependents', 'Number of Referrals',
    'Tenure in Months', 'Avg Monthly Long Distance Charges',
    'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges',
    'Total Refunds', 'Total Extra Data Charges',
    'Total Long Distance Charges', 'Satisfaction Score'
]

scaler = StandardScaler()

# Fit ONLY on training data, transform both
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print("Scaling complete.")
print(f"\nSample of scaled columns (first 3 rows of training set):")
print(X_train[scale_cols].head(3).round(3).to_string())
print(f"\nMean of scaled training columns (should be near 0):")
print(X_train[scale_cols].mean().round(3).to_string())
print(f"\nStd of scaled training columns (should be near 1):")
print(X_train[scale_cols].std().round(3).to_string())

Scaling complete.

Sample of scaled columns (first 3 rows of training set):
        Age  Number of Dependents  Number of Referrals  Tenure in Months  Avg Monthly Long Distance Charges  Avg Monthly GB Download  Monthly Charge  Total Charges  Total Refunds  Total Extra Data Charges  Total Long Distance Charges  Satisfaction Score
4626 -0.211                -0.484               -0.653            -0.670                             -1.115                   -0.810          -0.318         -0.622          -0.25                    -0.271                       -0.781               1.455
4192  0.919                 1.584               -0.653            -0.833                             -0.390                   -0.123           1.341         -0.446          -0.25                    -0.271                       -0.650              -0.205
5457 -0.866                -0.484               -0.653            -1.279                              1.700                   -0.172          -0.527         -0.98

## 8. Handling Class Imbalance
Computing class weights to handle the 73.5% / 26.5% imbalance during model training.
Instead of oversampling, we use class weighting which penalises the model more for missing churned customers.

In [9]:
classes = np.array([0, 1])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Class weights computed:")
print(f"  Not Churned (0): {class_weights[0]:.4f}")
print(f"  Churned (1):     {class_weights[1]:.4f}")
print(f"\nWhat this means:")
print(f"  Every churned customer mistake is penalised {class_weights[1]/class_weights[0]:.1f}x more than a non-churn mistake")

Class weights computed:
  Not Churned (0): 0.6806
  Churned (1):     1.8843

What this means:
  Every churned customer mistake is penalised 2.8x more than a non-churn mistake


## 9. Save Preprocessed Data
Saving all preprocessed outputs so the modelling notebook can load them directly without repeating preprocessing steps.

In [10]:
import joblib
import os

# Create a folder to store preprocessed outputs
os.makedirs('churnsight_artifacts', exist_ok=True)

# Save train and test sets
X_train.to_csv('churnsight_artifacts/X_train.csv', index=False)
X_test.to_csv('churnsight_artifacts/X_test.csv', index=False)
y_train.to_csv('churnsight_artifacts/y_train.csv', index=False)
y_test.to_csv('churnsight_artifacts/y_test.csv', index=False)

# Save scaler for use in Flask API later
joblib.dump(scaler, 'churnsight_artifacts/scaler.pkl')

# Save class weights
joblib.dump(class_weight_dict, 'churnsight_artifacts/class_weights.pkl')

# Save feature column list for API later
feature_cols = X_train.columns.tolist()
joblib.dump(feature_cols, 'churnsight_artifacts/feature_cols.pkl')

print("All artifacts saved to churnsight_artifacts/")
print(f"\nFiles saved:")
for f in os.listdir('churnsight_artifacts'):
    size = os.path.getsize(f'churnsight_artifacts/{f}')
    print(f"  {f} ({size:,} bytes)")

All artifacts saved to churnsight_artifacts/

Files saved:
  class_weights.pkl (144 bytes)
  feature_cols.pkl (876 bytes)
  scaler.pkl (1,415 bytes)
  X_test.csv (426,474 bytes)
  X_train.csv (1,702,729 bytes)
  y_test.csv (4,241 bytes)
  y_train.csv (16,916 bytes)


## 10. Preprocessing Summary

| Step | Action | Detail |
|---|---|---|
| Target Encoding | Yes/No to 1/0 | Churn Label converted to Churn Binary |
| Column Encoding | Binary mapping | 18 Yes/No columns mapped to 1/0 |
| Column Encoding | Ordinal encoding | Contract mapped to 0/1/2 (order preserved) |
| Column Encoding | One-hot encoding | Offer, Internet Type, Payment Method |
| Bool Fix | Bool to int | 13 one-hot columns converted from True/False to 1/0 |
| Train Test Split | 80/20 stratified | Churn rate preserved at 26.5% in both sets |
| Feature Scaling | StandardScaler | 12 numerical columns scaled (fit on train only) |
| Class Imbalance | Class weighting | Churned mistakes penalised 2.8x more |
| Artifacts Saved | joblib + csv | Scaler, weights, features, train/test sets saved |

**Output:** 44 features, 5,634 training samples, 1,409 test samples
**Next Stage:** Model Building and Evaluation